# Webscrape

In [8]:
import re
import requests
from bs4 import BeautifulSoup


def scrape_discovery(url: str) -> dict:
    r = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; research-scraper/1.0)"},
        timeout=30,
    )
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    page_text = soup.get_text("\n", strip=True)

    # --- location (authoritative) ---
    location = None
    loc_tag = soup.select_one(".page-title--sub-title.d-inline")
    if loc_tag:
        location = loc_tag.get_text(" ", strip=True)
        location = re.sub(r"\s+", " ", location).strip()

    # --- name ---
    name = None
    title_tag = soup.select_one(".page-title--title")
    if title_tag:
        name = title_tag.get_text(" ", strip=True)

    if not name:
        og = soup.select_one('meta[property="og:title"]')
        if og and og.get("content"):
            name = og["content"].strip()
        elif soup.title:
            name = soup.title.get_text(strip=True)

    if name:
        name = re.sub(r"\s+", " ", name)
        name = re.sub(r"\s*[\|\-–—]\s*Oervondstchecker\s*$", "", name).strip()

    # --- HARD REMOVE location from name ---
    if name and location:
        name = re.sub(re.escape(location), "", name, flags=re.IGNORECASE)
        name = re.sub(r"\s*[\|\-–—,:()]+\s*$", "", name)
        name = re.sub(r"\s{2,}", " ", name).strip()

    # --- date + finder name ("door <name>" / "by <name>") ---
    date = None
    found_by = None

    # Common on page: "Gevonden op 05-01-2026 door Justin van den Dool"
    m = re.search(
        r"Gevonden op\s+(\d{2}-\d{2}-\d{4})\s+door\s+(.+?)(?:\n|$)",
        page_text,
    )
    if m:
        date = m.group(1)
        found_by = m.group(2).strip()
    else:
        # Fallbacks
        m_date = re.search(r"Gevonden op\s+(\d{2}-\d{2}-\d{4})", page_text)
        date = m_date.group(1) if m_date else None

        m_by = re.search(
            r"(?:door|by)\s+(.+?)(?:\n|$)", page_text, flags=re.IGNORECASE
        )
        found_by = m_by.group(1).strip() if m_by else None

    # Clean up found_by if it accidentally includes extra tokens
    if found_by:
        found_by = re.sub(r"\s{2,}", " ", found_by).strip()
        # stop at common separators if they appear
        found_by = re.split(r"\s*(?:\||•|·)\s*", found_by)[0].strip()

    # --- description (skip the h3 heading) ---
    description = None
    desc_box = soup.select_one(".description.bg-light-grey.px-32.py-24.h-100")
    if desc_box:
        # Remove header(s) like <h3>Beschrijving</h3> / <h3>Description</h3>
        for h in desc_box.find_all(["h1", "h2", "h3", "h4"]):
            h.decompose()

        description = desc_box.get_text(" ", strip=True)
        description = re.sub(r"\s{2,}", " ", description).strip() or None

    # --- images (S3 large urls) ---
    image_urls = set()

    for img in soup.select("img[src]"):
        src = img["src"].strip()
        if "oervondstchecker.s3" in src and "/images/large/" in src:
            image_urls.add(src)

    for a in soup.select("a[href]"):
        href = a["href"].strip()
        if "oervondstchecker.s3" in href and "/images/large/" in href:
            image_urls.add(href)

    image_urls = sorted(image_urls)

    # --- expert name + notes (DEDUP + leaf-only to avoid "Geen foto" repetition) ---
    expert_name = None
    expert_notes = None

    exp_header = soup.find(
        lambda t: t.name in ("h2", "h3") and "Expert notities" in t.get_text()
    )

    if exp_header:
        header_text = exp_header.get_text(" ", strip=True)
        m2 = re.search(r"Expert notities:\s*(.+)$", header_text)
        expert_name = m2.group(1).strip() if m2 else None

        parts = []

        # Walk forward from header until next header; collect only leaf text nodes
        for sib in exp_header.next_elements:
            # stop at the next section header
            if getattr(sib, "name", None) in ("h2", "h3") and sib is not exp_header:
                break

            if getattr(sib, "name", None) in ("p", "li", "span", "div"):
                # leaf-only: skip containers that wrap other text blocks
                if sib.find(["p", "li", "span", "div"]):
                    continue

                t = sib.get_text(" ", strip=True)
                if t:
                    t = re.sub(r"\s{2,}", " ", t).strip()
                    parts.append(t)

        # de-dup consecutive repeats (common with nested rendering)
        dedup = []
        for t in parts:
            if not dedup or dedup[-1] != t:
                dedup.append(t)

        # if it's the same token repeated (e.g., "Geen foto"), keep one
        if dedup and len(set(dedup)) == 1:
            expert_notes = dedup[0]
        else:
            expert_notes = " ".join(dedup).strip() if dedup else None

    return {
        "date": date,
        "name": name,
        "location": location,
        "found_by": found_by,
        "description": description,
        "image_urls": image_urls,
        "expert_name": expert_name,
        "expert_notes": expert_notes,
        "source_url": url,
    }


# Example
url = "https://www.oervondstchecker.nl/discoveries/view/000fa8f3-26b1-44ce-a1f8-7e82a5509c07?guid=000fa8f3-26b1-44ce-a1f8-7e82a5509c07&q="
row = scrape_discovery(url)
row


{'date': '21-04-2025',
 'name': 'Kies',
 'location': 'van??',
 'found_by': 'Erriegraven',
 'description': 'Hi vandaag gevonden op de Zandmotor. Kies van ? Dacht eerst aan zwijn, maar ik mis bepaalde kenmerken. Beer misschien?',
 'image_urls': [],
 'expert_name': 'Charlie Schouwenburg',
 'expert_notes': 'Geen foto',
 'source_url': 'https://www.oervondstchecker.nl/discoveries/view/000fa8f3-26b1-44ce-a1f8-7e82a5509c07?guid=000fa8f3-26b1-44ce-a1f8-7e82a5509c07&q='}

In [9]:
from playwright.async_api import async_playwright

START_URL = "https://www.oervondstchecker.nl/alle-vondsten?guid=&q="

async def get_discovery_links():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(viewport={"width": 1280, "height": 800})
        await page.goto(START_URL, wait_until="networkidle")

        last_count = 0
        stable_rounds = 0
        max_stable_rounds = 50000
        pause_ms = 1200

        while True:
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await page.wait_for_timeout(pause_ms)

            count = await page.evaluate("() => document.querySelectorAll('div.discovery[id]').length")
            if count == last_count:
                stable_rounds += 1
            else:
                stable_rounds = 0
                last_count = count

            if stable_rounds >= max_stable_rounds:
                break

        guids = await page.evaluate("""
            () => Array.from(document.querySelectorAll('div.discovery[id]'))
                .map(el => el.id)
                .filter(Boolean)
        """)

        await browser.close()

    guids = sorted(set(guids))
    links = [f"https://www.oervondstchecker.nl/discoveries/view/{g}?guid={g}&q=" for g in guids]
    return links

# ✅ this is the key change:
discovery_links = await get_discovery_links()

len(discovery_links), discovery_links[:5]

ModuleNotFoundError: No module named 'playwright'

In [ ]:
with open("discovery_guids.txt", "w", encoding="utf-8") as f:
    for url in discovery_links:
        f.write(url.split("/")[-1].split("?")[0] + "\n")


In [ ]:
with open("discovery_guids.txt", "r", encoding="utf-8") as f:
    guids = [line.strip() for line in f if line.strip()]

links = [
    f"https://www.oervondstchecker.nl/discoveries/view/{g}?guid={g}&q="
    for g in guids
]
links

['https://www.oervondstchecker.nl/discoveries/view/000a00ce-a567-4082-8566-64e2e0460401?guid=000a00ce-a567-4082-8566-64e2e0460401&q=',
 'https://www.oervondstchecker.nl/discoveries/view/000b3447-8f91-4373-a100-d401a6e443bb?guid=000b3447-8f91-4373-a100-d401a6e443bb&q=',
 'https://www.oervondstchecker.nl/discoveries/view/000fa8f3-26b1-44ce-a1f8-7e82a5509c07?guid=000fa8f3-26b1-44ce-a1f8-7e82a5509c07&q=',
 'https://www.oervondstchecker.nl/discoveries/view/001e59fb-94ad-44a4-b185-a38dd6ada6d5?guid=001e59fb-94ad-44a4-b185-a38dd6ada6d5&q=',
 'https://www.oervondstchecker.nl/discoveries/view/001f4dc6-6e01-48a3-b68a-62d4a7a6cd30?guid=001f4dc6-6e01-48a3-b68a-62d4a7a6cd30&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002bcf43-7ccf-4171-b3c9-923f43b02e92?guid=002bcf43-7ccf-4171-b3c9-923f43b02e92&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002c02fe-fa86-49a7-86b7-07009930977d?guid=002c02fe-fa86-49a7-86b7-07009930977d&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002c

In [ ]:
discovery_links

['https://www.oervondstchecker.nl/discoveries/view/000a00ce-a567-4082-8566-64e2e0460401?guid=000a00ce-a567-4082-8566-64e2e0460401&q=',
 'https://www.oervondstchecker.nl/discoveries/view/000b3447-8f91-4373-a100-d401a6e443bb?guid=000b3447-8f91-4373-a100-d401a6e443bb&q=',
 'https://www.oervondstchecker.nl/discoveries/view/000fa8f3-26b1-44ce-a1f8-7e82a5509c07?guid=000fa8f3-26b1-44ce-a1f8-7e82a5509c07&q=',
 'https://www.oervondstchecker.nl/discoveries/view/001e59fb-94ad-44a4-b185-a38dd6ada6d5?guid=001e59fb-94ad-44a4-b185-a38dd6ada6d5&q=',
 'https://www.oervondstchecker.nl/discoveries/view/001f4dc6-6e01-48a3-b68a-62d4a7a6cd30?guid=001f4dc6-6e01-48a3-b68a-62d4a7a6cd30&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002bcf43-7ccf-4171-b3c9-923f43b02e92?guid=002bcf43-7ccf-4171-b3c9-923f43b02e92&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002c02fe-fa86-49a7-86b7-07009930977d?guid=002c02fe-fa86-49a7-86b7-07009930977d&q=',
 'https://www.oervondstchecker.nl/discoveries/view/002c

In [ ]:
rows = []
for u in discovery_links:
    try:
        rows.append(scrape_discovery(u))
    except Exception as e:
        print("Failed:", u, e)


In [ ]:
output = '2026-01-07Data-exportOervondstchecker.parquet'
df.to_parquet(output)

In [11]:
import pandas as pd
df = pd.read_parquet("2026-01-07Data-exportOervondstchecker.parquet")


Translate

In [12]:
df["expert_notes"]

0        Klopt en zo groot kan het alleen maar een frag...
1                      Dit is een scheenbeen van een paard
2                                                Geen foto
3                                  Klopt, mammoetslagtand.
4        Het lijkt er op, maar voor zover ik kan zien i...
                               ...                        
31624    Zulke fragmenten blijven moeilijk. Waarschijnl...
31625    Een fragment van een rib en gezien het formaat...
31626      Dit is een stuk kaak van een wolharige mammoet.
31627    Klopt, twee laatste teenkoten. Beiden van een ...
31628    Er is veel mee gebeurd, maar ik denk allemaal ...
Name: expert_notes, Length: 31629, dtype: object

In [13]:
df["image_urls"].to_list()

[array(['https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/2db02f1d-a1ec-4a64-b409-5fa95d71176e.jpg',
        'https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/79c97810-9616-4543-9a0d-6125650a3084.jpg',
        'https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/8fa76e13-861a-448d-8753-daff9c5eed1c.jpg'],
       dtype=object),
 array(['https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/c607f5d0-3faa-4d4a-82df-8cd45b59b67f.jpg',
        'https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/ded23e34-d4a2-44b3-a99d-e271b6b303ce.jpg'],
       dtype=object),
 array([], dtype=object),
 array(['https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/28b38868-3550-40dc-bbc0-35cc95f3600c.jpg',
        'https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/81697680-6d7c-4508-9fbd-6977b3019be1.jpg',
        'https://oervondstchecker.s3.eu-central-1.amazonaws.com/images/large/846907c8-7d

In [14]:
df['date'] = pd.to_datetime(df['date'])
df.groupby('date')['source_url'].count().sort_index()


/var/folders/_p/r6dt32s51pgc2j9lb94tb5qh0000gn/T/ipykernel_905/3542626774.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date'] = pd.to_datetime(df['date'])


date
2014-01-24    15
2014-01-25    13
2014-01-26     7
2014-01-27    12
2014-01-28    12
              ..
2026-01-03     4
2026-01-04     4
2026-01-05     3
2026-01-06     3
2026-01-07     6
Name: source_url, Length: 4054, dtype: int64

In [15]:
INCLUDE = ['name', 'description','expert_notes']
text_cols = df.columns
text_cols = [c for c in text_cols if c in INCLUDE]
text_cols

['name', 'description', 'expert_notes']

In [16]:
import pandas as pd
from functools import lru_cache
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

@lru_cache(maxsize=200_000)
def translate_nl_en(text: str) -> str:
    text = str(text).strip()
    if not text:
        return text
    # tokenize + translate
    batch = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt")
    out = model.generate(**batch)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

def translate_df_columns_free(data: pd.DataFrame, text_cols: list[str]) -> pd.DataFrame:
    # 1) stack unique strings
    s = data[text_cols].stack(dropna=False)
    mask = s.map(type).eq(str)
    uniq = pd.unique(s[mask])

    print(f"Unique strings to translate: {len(uniq):,}")

    # 2) build map
    trans_map = {u: translate_nl_en(u) for u in uniq}

    # 3) vectorized remap
    data[text_cols] = data[text_cols].apply(lambda col: col.map(lambda v: trans_map.get(v, v)))
    return data


/Users/georgianamg93gmail.com/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [17]:
df = translate_df_columns_free(df, text_cols)   # MarianMT version

/var/folders/_p/r6dt32s51pgc2j9lb94tb5qh0000gn/T/ipykernel_905/1131702349.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  s = data[text_cols].stack(dropna=False)
/Users/georgianamg93gmail.com/Library/Python/3.9/lib/python/site-packages/transformers/tokenization_utils_base.py:4291: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs

Unique strings to translate: 66,918


In [18]:
output = '2026-01-07Data-exportOervondstchecker_translated.parquet'
df.to_parquet(output)

In [19]:
df

,date,name,location,found_by,description,image_urls,expert_name,expert_notes,source_url
0,2024-05-15,Is this a,bot,Suzan waaijer,So this could be a bone.,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,True and that big can only be a fragment of on...,https://www.oervondstchecker.nl/discoveries/vi...
1,2022-02-03,Large bone,None,Kim,Any ideas?,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,This is a tibia of a horse.,https://www.oervondstchecker.nl/discoveries/vi...
2,2025-04-21,Choose,van??,Erriegraven,Found it today on the Sand Engine. Choose from...,[],Charlie Schouwenburg,No photo,https://www.oervondstchecker.nl/discoveries/vi...
3,2024-09-01,Mammoet,slagtand,Léon Smiers,"Hello, I found this beautiful mammoth tusks on...",[https://oervondstchecker.s3.eu-central-1.amaz...,Hansjorg Ahrens,"That's right, mammoth tusk.",https://www.oervondstchecker.nl/discoveries/vi...
4,2022-02-11,Petrified,Hout?,Marco,Found it on the 2nd Maasvlakte.,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,"Looks like it, but as far as I can tell, this ...",https://www.oervondstchecker.nl/discoveries/vi...
...,...,...,...,...,...,...,...,...,...
31624,2020-12-29,Piece,bot,Marlou S.,Any idea what this piece of bone could be?,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,"Such fragments remain difficult, probably from...",https://www.oervondstchecker.nl/discoveries/vi...
31625,2021-05-07,Bone,None,Ilja Zeilstra,What animal is this bone from?,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,A fragment of a rib and given the size of a ma...,https://www.oervondstchecker.nl/discoveries/vi...
31626,2023-09-11,Archological,vondst,B.Vrijhof- v.d. Gevel,It's a heavy piece of bone.,[https://oervondstchecker.s3.eu-central-1.amaz...,Hester Loeff,This is a piece of jaw from a wooly mammoth.,https://www.oervondstchecker.nl/discoveries/vi...
31627,2021-08-16,Two,kootjes,Jan,Is there any way to find out which animal?,[https://oervondstchecker.s3.eu-central-1.amaz...,Charlie Schouwenburg,"That's right, two last toeholes, both of them ...",https://www.oervondstchecker.nl/discoveries/vi...
